# データベース設計と設定(Day2)

## データベース設計の本質
```
DB設計は「とりあえず保存できる」ことより、あとで安全に使い続けられること
```
## データベースの導入目的
目的がないのに導入するのは学習目的ということ以外に価値をなさない。
```
- JEPXの価格データを蓄積する
- 後で期間・エリアで抽出する
- 分析や可視化に使う
- 将来、気象データやスパイク検出結果と結合する
```
## 設計の方針
### 責務の分離

ex)
- エリア名などの固定情報
- 時刻ごとの価格データ

### データ情報の明確化
1行 = なんの記録かを言葉で説明できる設計にする

### レコードの識別方法
重複はないですか？

### 型の決定
日時・数値・文字列などは意味に合った型を使う

### 正規化しなさすぎ、しすぎ
冗長データを減らすために正規化するが、分けすぎても使いにくい

### 検索方法を考える
よく検索する列にはインデックスを付けることがおすすめ

### データ整合性をDBで守る
NOT NULL、FOREIGN KEY、UNIQUEなどの制約

## DB設計
### 作る設計書
- テーブル一覧
- 各テーブルのカラム定義
- 主キー・外部キー・一意制約・インデックス

### テーブル一覧
| テーブル名 | 論理名 | 役割 |
|---|---|---|
| areas | エリアマスタ | エリアコード・エリア名を管理する |
| jepx_prices | JEPX価格データ | 30分ごとのエリア別価格を保存する |

### 各テーブルのカラム定義
#### areas

**役割**  
エリア情報を管理するマスタテーブル。  
価格テーブルや将来の気象・需要データとの結合基準として用いる。

| カラム名 | データ型 | NULL | PK | UK | FK | 説明 |
|---|---|---|---|---|---|---|
| area_id | SERIAL | NOT NULL | ○ |  |  | エリアID |
| area_code | VARCHAR(20) | NOT NULL |  | ○ |  | エリアコード（tokyo, kansai など） |
| area_name_ja | VARCHAR(50) | NOT NULL |  |  |  | 日本語エリア名 |
| area_name_en | VARCHAR(50) | NOT NULL |  |  |  | 英語エリア名 |

**制約**
- PRIMARY KEY: `area_id`
- UNIQUE: `area_code`

#### jepx_prices

**役割**  
JEPXの30分価格データをエリア別・時刻別に保存する事実テーブル。  
分析の中心となるテーブル。

| カラム名 | データ型 | NULL | PK | UK | FK | 説明 |
|---|---|---|---|---|---|---|
| price_id | BIGSERIAL | NOT NULL | ○ |  |  | 価格ID |
| delivery_date | DATE | NOT NULL |  |  |  | 受渡日 |
| slot | SMALLINT | NOT NULL |  |  |  | 時刻コード（1〜48） |
| dt_start | TIMESTAMP | NOT NULL |  |  |  | 30分コマ開始日時 |
| area_id | INTEGER | NOT NULL |  |  | ○ | エリアID |
| price_yen_per_kwh | NUMERIC(10,4) | NOT NULL |  |  |  | 価格（円/kWh） |
| created_at | TIMESTAMP | NOT NULL |  |  |  | 登録日時 |

**制約**
- PRIMARY KEY: `price_id`
- FOREIGN KEY: `area_id` → `areas.area_id`
- UNIQUE: `(dt_start, area_id)`

**一意制約の意味**  
同一日時・同一エリアの価格データは1件のみとする。  
CSVの再取込時の重複登録防止にも利用する。

**格納イメージ**
| delivery_date | slot | dt_start            | area_id | price_yen_per_kwh |
| ------------- | ---: | ------------------- | ------: | ----------------: |
| 2023-04-01    |    1 | 2023-04-01 00:00:00 |       1 |              12.3 |
| 2023-04-01    |    1 | 2023-04-01 00:00:00 |       2 |              10.8 |
| 2023-04-01    |    1 | 2023-04-01 00:00:00 |       3 |               9.7 |


### インデックス設計
>データベースにおけるインデックスは、テーブル内のデータの検索やソートを高速化するために使用されるデータ構造です。
>
>インデックスは特定の列（または複数の列）に対して作成され、その列の値とそれに対応するデータの物理的な位置情報を持ちます。
>
>これにより、データベースエンジンはインデックスを使用して効率的なデータの検索や結合を行うことができます。
>
>[DBにおけるインデックス設計入門](https://zenn.dev/yuki_tu/articles/c0a11385a33e0c)

#### インデックス設計の考え方
「どんなSQLが走るか」から考えると作りやすい

例えば、設計したテーブルについて以下のような抽出をする

```sql
-- 期間で抽出
SELECT *
FROM jepx_prices
WHERE dt_start BETWEEN '2023-09-01' AND '2023-10-01';

-- エリアで抽出
SELECT *
FROM jepx_prices
WHERE area_id = 3;

-- エリア×期間
SELECT *
FROM jepx_prices
WHERE area_id = 3
AND dt_start BETWEEN '2023-09-01' AND '2023-10-01';
```

これらが高速で検索できるためにはどうすべきかと問いを立て、例えば少数のデータを抽出するときに全検索せずに、元から搾れるものは絞って検索するなど工夫する

#### 作成インデックス

| インデックス名                       | 対象テーブル      | 対象カラム               | 目的          |
| ----------------------------- | ----------- | ------------------- | ----------- |
| idx_jepx_prices_dt_start      | jepx_prices | dt_start            | 期間検索高速化     |
| idx_jepx_prices_area_id       | jepx_prices | area_id             | エリア検索高速化    |
| idx_jepx_prices_delivery_date | jepx_prices | delivery_date       | 日付検索高速化     |
| idx_jepx_prices_area_dt       | jepx_prices | (area_id, dt_start) | エリア×期間検索高速化 |

### DDL(Data Definition Language)

```sql
CREATE TABLE areas (
    area_id SERIAL PRIMARY KEY,
    area_code VARCHAR(20) UNIQUE NOT NULL,
    area_name_ja VARCHAR(50) NOT NULL,
    area_name_en VARCHAR(50) NOT NULL
);

CREATE TABLE jepx_prices (
    price_id BIGSERIAL PRIMARY KEY,
    delivery_date DATE NOT NULL,
    slot SMALLINT NOT NULL,
    dt_start TIMESTAMP NOT NULL,
    area_id INTEGER NOT NULL REFERENCES areas(area_id),
    price_yen_per_kwh NUMERIC(10,4) NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (dt_start, area_id)
);
```

なお、データ操作に関するコマンドはDML（Data Manipulation Language）と呼ぶ
```SQL
CREATE INDEX idx_jepx_prices_dt_start ON jepx_prices(dt_start);
CREATE INDEX idx_jepx_prices_area_id ON jepx_prices(area_id);
CREATE INDEX idx_jepx_prices_delivery_date ON jepx_prices(delivery_date);
CREATE INDEX idx_jepx_prices_area_dt ON jepx_prices(area_id, dt_start);
```

# 環境構築
pstgresqlインストール
```bash
sudo apt update
sudo apt install -y postgresql postgresql-contrib
```

起動
```bash
sudo systemctl start postgresql
sudo systemctl enable postgresql
```

起動確認
```bash
sudo systemctl status postgresql
```

psql　(対話型コマンドライン・フロントエンドツール)に入る
```bash
sudo -u postgres psql
```

# database構築
## sql基本コマンド

データベース一覧
```SQL
\l
```

DB接続
```SQL
\c jepx_analysis
```

テーブル一覧
```SQL
\dt
```

psql終了
```SQL
\q
```
## DB作成
データベースjepx_analysis、DBユーザーjepx_user、jepx_userへのフル権限付与

```SQL
CREATE USER jepx_user WITH PASSWORD 'jepx_pass';
CREATE DATABASE jepx_analysis OWNER jepx_user;
GRANT ALL PRIVILEGES ON DATABASE jepx_analysis TO jepx_user;
```

テーブル作成

```SQL
CREATE TABLE areas (
    area_id SERIAL PRIMARY KEY,
    area_code VARCHAR(20) UNIQUE NOT NULL,
    area_name_ja VARCHAR(50) NOT NULL,
    area_name_en VARCHAR(50) NOT NULL
);

CREATE TABLE jepx_prices (
    price_id BIGSERIAL PRIMARY KEY,
    delivery_date DATE NOT NULL,
    slot SMALLINT NOT NULL,
    dt_start TIMESTAMP NOT NULL,
    area_id INTEGER NOT NULL REFERENCES areas(area_id),
    price_yen_per_kwh NUMERIC(10,4) NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (dt_start, area_id)
);
```

エリアマスタ
```SQL
INSERT INTO areas (area_code, area_name_ja, area_name_en) VALUES
('hokkaido', '北海道', 'Hokkaido'),
('tohoku', '東北', 'Tohoku'),
('tokyo', '東京', 'Tokyo'),
('chubu', '中部', 'Chubu'),
('hokuriku', '北陸', 'Hokuriku'),
('kansai', '関西', 'Kansai'),
('chugoku', '中国', 'Chugoku'),
('shikoku', '四国', 'Shikoku'),
('kyushu', '九州', 'Kyushu');
```

# 接続確認
## 使用ライブラリ
| ライブラリ      | 役割                  |
| ---------- | ------------------- |
| psycopg2   | PostgreSQL接続ドライバ    |
| SQLAlchemy | PythonでDBを扱うフレームワーク |

```bash
pip install psycopg2-binary sqlalchemy pandas
```

## test_db.py
接続（具体的な値はconfig.py）
```python
postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}
```
意味
```bash
DB種類 + ドライバ://ユーザー:パスワード@サーバー:ポート/データベース
```

```
どのDB？
どのドライバ？
誰が？
どこに？
どのDBに？
```

ポート番号の確認
```bash
sudo ss -plnt | grep postgres
```

```text
LISTEN 0 128 127.0.0.1:5432
```